# 📘 W4 Python Lab — 차트 Vision + Claude 3D 에이전트

**n8n W4 강의의 Python 버전.** 차트 이미지를 Claude Vision으로 분석해 3D 에이전트를 완성합니다.

## 학습 목표
- Chart-IMG API로 TradingView 차트 PNG 다운로드
- Claude Vision (멀티모달 API)으로 차트 패턴 자동 판독
- W1 (가격) + W3 (뉴스) + W4 (차트) 통합 → 3D verdict 생성
- Discord Webhook으로 차트 + 답변 동시 발송

## 🛠 사전 준비

```bash
pip install anthropic requests python-dotenv pillow
```

`.env` 추가:
```
CHART_IMG_KEY=여러분의_Chart-IMG_키
ANTHROPIC_API_KEY=여러분의_Claude_키
DISCORD_WEBHOOK_URL=Discord_채널_웹훅_URL
```

> 💡 **n8n과 비교:** `Analyze Image` 노드 = `anthropic.messages.create()` with image block. 동일한 Vision API를 두 방식으로 호출.

## 1. 라이브러리 import + 환경 로드

In [ ]:
import os
import json
import base64
import requests
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

CHART_IMG_KEY = os.getenv('CHART_IMG_KEY')
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
DISCORD_WEBHOOK_URL = os.getenv('DISCORD_WEBHOOK_URL')

client = Anthropic(api_key=ANTHROPIC_API_KEY)
print(f'Chart-IMG: {"✓" if CHART_IMG_KEY else "✗"}')
print(f'Claude: {"✓" if ANTHROPIC_API_KEY else "✗"}')
print(f'Discord: {"✓" if DISCORD_WEBHOOK_URL else "✗"}')

## 2. Chart-IMG로 차트 PNG 다운로드

TradingView 차트를 API로 받아옵니다. 무료 500회/월.

In [ ]:
def download_chart(symbol: str, interval: str = '1D', 
                   save_dir: str = 'charts') -> str:
    """TradingView 차트를 PNG로 저장 후 파일 경로 반환.
    
    Args:
        symbol: 'NASDAQ:AAPL', 'KRX:005930', 'BINANCE:BTCUSDT' 등
        interval: '15m', '1h', '1D', '1W' 등
    """
    url = 'https://api.chart-img.com/v2/tradingview/advanced-chart'
    headers = {'x-api-key': CHART_IMG_KEY, 'content-type': 'application/json'}
    payload = {
        'symbol': symbol,
        'interval': interval,
        'theme': 'dark',
        'width': 800,
        'height': 500,
        'studies': [
            {'name': 'Relative Strength Index', 'input': {'length': 14}},
            {'name': 'Moving Average', 'input': {'length': 20}}
        ]
    }
    
    response = requests.post(url, json=payload, headers=headers, timeout=20)
    response.raise_for_status()
    
    # PNG 저장
    Path(save_dir).mkdir(exist_ok=True)
    safe_name = symbol.replace(':', '_')
    filename = f'{save_dir}/{safe_name}_{datetime.now():%Y%m%d_%H%M}.png'
    with open(filename, 'wb') as f:
        f.write(response.content)
    
    print(f'✓ Saved: {filename} ({len(response.content) / 1024:.1f} KB)')
    return filename

# 테스트: 애플 차트
chart_path = download_chart('NASDAQ:AAPL', '1D')

## 3. Claude Vision으로 차트 분석

핵심: 이미지를 base64로 인코딩해서 Claude에 전달.

In [ ]:
def image_to_base64(image_path: str) -> str:
    """이미지 파일을 base64 문자열로 변환."""
    with open(image_path, 'rb') as f:
        return base64.standard_b64encode(f.read()).decode('utf-8')

def analyze_chart(image_path: str, ticker: str) -> dict:
    """Claude Vision으로 차트 패턴 판독."""
    img_b64 = image_to_base64(image_path)
    
    prompt = f"""이 차트는 {ticker}의 일봉입니다. 다음 항목을 JSON으로 답하세요:

1. trend: UPTREND / DOWNTREND / SIDEWAYS
2. pattern: 발견된 패턴 (헤드앤숄더, 삼각수렴, 이중바닥, 돌파, 조정 등) 또는 NONE
3. rsi_zone: OVERSOLD (RSI<30) / NEUTRAL (30-70) / OVERBOUGHT (>70)
4. signal: BUY / SELL / HOLD
5. confidence: 1-5
6. reasoning: 한 문장 근거

JSON만 출력. 설명 금지."""
    
    message = client.messages.create(
        model='claude-haiku-4-5',
        max_tokens=500,
        messages=[{
            'role': 'user',
            'content': [
                {
                    'type': 'image',
                    'source': {
                        'type': 'base64',
                        'media_type': 'image/png',
                        'data': img_b64
                    }
                },
                {'type': 'text', 'text': prompt}
            ]
        }]
    )
    
    # JSON 파싱 (Claude가 마크다운 코드블록으로 감쌀 수 있음)
    text = message.content[0].text.strip()
    if text.startswith('```'):
        text = text.split('```')[1].replace('json\n', '').strip()
    
    return json.loads(text)

# 테스트
chart_analysis = analyze_chart(chart_path, 'AAPL')
print(json.dumps(chart_analysis, indent=2, ensure_ascii=False))

## 4. W1~W3 결과 시뮬레이션 (실제로는 앞 주차에서 받아옴)

n8n에서는 Merge 노드로 합쳤습니다. Python에서는 딕셔너리 합치기로 간단히.

In [ ]:
# 예시: W1~W3 결과 (실제로는 해당 주차 함수 재사용)
mock_w1 = {
    'price': 178.42,
    'change_pct': 1.23
}
mock_w2 = {
    'rsi14': 62.5,
    'ma20': 175.10,
    'bb_upper': 182.5,
    'bb_lower': 168.3
}
mock_w3 = {
    'sentiment': 0.18,
    'top_headline': 'Apple announces new AI chip at WWDC',
    'article_count': 8
}

# W4 결과 (방금 받은 차트 분석)
w4 = chart_analysis

# 통합 딕셔너리 (n8n Merge 노드 역할)
combined = {
    'ticker': 'AAPL',
    'dimension_1_price': mock_w1,
    'dimension_2_news': mock_w3,
    'dimension_3_chart': w4,
    'indicators': mock_w2,
    'timestamp': datetime.now().isoformat()
}

print(json.dumps(combined, indent=2, ensure_ascii=False))

## 5. 3D 에이전트 — 최종 판단

세 차원을 모두 본 후 Claude가 최종 verdict 생성.

In [ ]:
def agent_3d_verdict(data: dict) -> dict:
    """3D 분석 종합 → 최종 투자 판단."""
    
    system = """당신은 투자 분석 에이전트입니다. 3차원 입력(가격·뉴스·차트)을 받아 판단합니다.
규칙:
- 3차원이 모두 일치하면 confidence 5
- 2차원이 일치하면 confidence 3~4
- 차원 간 상충되면 WATCH + 낮은 confidence
- 교육 목적 분석임을 명심

출력 JSON 스키마:
{"verdict": "BUY|SELL|WATCH|HOLD", "confidence": 1-5, 
 "reasoning": "2문장", "consensus": "ALIGNED|MIXED"}

JSON만 출력."""
    
    user = f'다음 데이터를 분석하세요:\n\n{json.dumps(data, ensure_ascii=False, indent=2)}'
    
    message = client.messages.create(
        model='claude-sonnet-4-6',
        max_tokens=500,
        system=system,
        messages=[{'role': 'user', 'content': user}]
    )
    
    text = message.content[0].text.strip()
    if text.startswith('```'):
        text = text.split('```')[1].replace('json\n', '').strip()
    
    return json.loads(text)

verdict = agent_3d_verdict(combined)
print('🎯 Final Verdict:')
print(json.dumps(verdict, indent=2, ensure_ascii=False))

## 6. Discord 발송 — 차트 + 답변 한 번에

n8n Discord 노드 = `requests.post()` with multipart.

In [ ]:
def send_to_discord(verdict: dict, chart_path: str, ticker: str):
    """verdict + 차트 이미지를 Discord에 발송."""
    if not DISCORD_WEBHOOK_URL:
        print('⚠ Discord Webhook 미설정')
        return
    
    # 색상: BUY 초록 / SELL 빨강 / WATCH 노랑 / HOLD 회색
    colors = {'BUY': 0x22c55e, 'SELL': 0xef4444, 'WATCH': 0xeab308, 'HOLD': 0x6b7280}
    color = colors.get(verdict['verdict'], 0x6b7280)
    
    payload = {
        'embeds': [{
            'title': f'🤖 {ticker} 3D 분석',
            'color': color,
            'fields': [
                {'name': 'Verdict', 'value': verdict['verdict'], 'inline': True},
                {'name': 'Confidence', 'value': f"{verdict['confidence']}/5", 'inline': True},
                {'name': 'Consensus', 'value': verdict.get('consensus', 'N/A'), 'inline': True},
                {'name': 'Reasoning', 'value': verdict['reasoning']}
            ],
            'image': {'url': 'attachment://chart.png'},
            'footer': {'text': '교육 목적 · 투자 권유 아님'}
        }]
    }
    
    # multipart로 이미지 첨부
    with open(chart_path, 'rb') as f:
        files = {
            'payload_json': (None, json.dumps(payload)),
            'files[0]': ('chart.png', f, 'image/png')
        }
        response = requests.post(DISCORD_WEBHOOK_URL, files=files, timeout=10)
    
    if response.status_code in [200, 204]:
        print('✓ Discord 발송 완료')
    else:
        print(f'✗ Discord 에러: {response.status_code} {response.text}')

# 실행
send_to_discord(verdict, chart_path, 'AAPL')

## 7. 전체 파이프라인 — 한 번에 실행

지금까지 만든 함수를 한 줄로 엮습니다. 이게 W4 완성본.

In [ ]:
def run_3d_pipeline(ticker: str, chart_symbol: str):
    """3D 에이전트 전체 파이프라인 실행.
    
    Args:
        ticker: 'AAPL' (Claude 프롬프트용 약칭)
        chart_symbol: 'NASDAQ:AAPL' (Chart-IMG용 전체 심볼)
    """
    print(f'\n=== 🚀 {ticker} 3D 분석 시작 ===\n')
    
    # 1. 차트 다운로드
    chart_path = download_chart(chart_symbol)
    
    # 2. Claude Vision 분석
    chart_result = analyze_chart(chart_path, ticker)
    print(f'✓ 차트 분석: {chart_result["signal"]} ({chart_result["pattern"]})')
    
    # 3. (실제론 W1~W3 결과 받아와야 함 — 여기선 mock)
    combined_data = {
        'ticker': ticker,
        'chart': chart_result,
        'price': mock_w1,
        'news': mock_w3,
        'indicators': mock_w2
    }
    
    # 4. 3D verdict
    verdict = agent_3d_verdict(combined_data)
    print(f'✓ 최종 판단: {verdict["verdict"]} (conf {verdict["confidence"]})')
    
    # 5. Discord 발송
    send_to_discord(verdict, chart_path, ticker)
    
    return verdict

# 실행
result = run_3d_pipeline('AAPL', 'NASDAQ:AAPL')

## 🎯 과제

1. **멀티 종목 확장** — AAPL, MSFT, NVDA 3개 종목을 for 루프로 순차 분석
2. **실제 W1~W3 연결** — mock 딕셔너리 대신 앞 주차 함수를 직접 호출해 실데이터로 통합
3. **이미지 리사이즈** — `Pillow`로 차트를 400x250으로 축소해 Vision API 호출 비용 절감 (≈60% 절약)
4. **Switch 로직 Python 구현** — verdict에 따라 다른 채널 발송 (BUY→긴급, WATCH→일반)
   ```python
   if verdict['verdict'] in ['BUY', 'SELL'] and verdict['confidence'] >= 4:
       send_to_discord(verdict, chart_path, ticker)  # 긴급 채널
   else:
       log_only(verdict)  # 기록만
   ```
5. **비용 측정** — `message.usage`에서 input/output 토큰 수 추출 후 1회 호출 비용 계산

## 🔄 n8n vs Python 비교

| 개념 | n8n | Python |
|---|---|---|
| Chart-IMG 호출 | HTTP Request + binary | `requests.post` + `.content` |
| Vision 분석 | Anthropic Analyze Image 노드 | `client.messages.create` with image block |
| Merge (W1~W4) | Merge 노드 | 딕셔너리 병합 |
| 3D 에이전트 | AI Agent 노드 | `client.messages.create` + system prompt |
| Discord 첨부 | Discord 노드 | `requests.post` with multipart |
| Switch 분기 | Switch 노드 | `if-elif-else` |

**핵심 배움:** Vision API는 이미지를 **base64 문자열로 변환**해서 텍스트 프롬프트와 함께 전달합니다. n8n이 이 작업을 자동화해주는 것뿐 — 실제 원리는 동일.